# SwarmSort Phase 2b: proxy-validity check

Is the cheap proxy training (12 epochs, stratified 40% of the train split) a faithful stand-in for longer training?
Eight configurations (7 Latin-hypercube samples from the search space + the Ultralytics defaults) are trained with
the proxy and with a medium setting (40 epochs, full train split); every run uses imgsz 416. The proxy passes if the Spearman rank
correlation of their val mAP@50 is at least 0.6.

Output: `/kaggle/working/SwarmSort/results/proxy_check.json` (every run, rho, p-value, verdict) and `results/runs/`.

**Settings:** Accelerator = GPU, Internet = on, Input = the *Garbage Detection - 6 Waste Categories* dataset.
Run it with **Save Version -> Save & Run All (Commit)** so it keeps running with the browser closed.

**Use the same Kaggle accelerator type (e.g. always GPU P100) for every phase**, so run times are comparable and
results are not affected by GPU-specific numerics. Every JSON output records the GPU name.

### Resuming after a dead session

`proxy_check.json` is rewritten after every finished run, and finished runs are skipped on restart, so a dead session
loses at most one run. To continue: open the notebook, **Add Input -> Notebook Output** and pick the previous
(failed or timed-out) version of this notebook, then run it again. The cell below copies that version's
`results/proxy_check.json` into place before the experiment starts.

In [ ]:
import os
REPO = "/kaggle/working/SwarmSort"
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/Shreyansh303/SwarmSort.git {REPO}
%cd {REPO}
# Kaggle's own CUDA build of torch is kept: only Ultralytics is installed, at the version pinned in requirements.txt
!pip install -q ultralytics==8.4.169
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# Rebuild configs/data.yaml and configs/lists/ with Kaggle paths from the committed configs/split.csv
!python src/verify_dataset.py --root /kaggle/input
assert os.path.exists("configs/data.yaml"), "verify_dataset.py did not write configs/data.yaml"
print(open("configs/data.yaml").read())

In [ ]:
# Resume: reuse the log of a previous version of this notebook if one is attached as input
import glob, shutil
previous = sorted(glob.glob("/kaggle/input/**/results/proxy_check.json", recursive=True))
if previous and not os.path.exists("results/proxy_check.json"):
    os.makedirs("results", exist_ok=True)
    shutil.copy(previous[0], "results/proxy_check.json")
    print("Resuming from", previous[0])
else:
    print("Starting a new log" if not previous else "results/proxy_check.json already present")

In [ ]:
# Stratified 40% subset of the train split -> configs/lists/train_proxy.txt and configs/data_proxy.yaml
!python src/training.py --data configs/data.yaml --fraction 0.4

In [ ]:
!python src/proxy_check.py --device 0

In [ ]:
import json
log = json.load(open("results/proxy_check.json"))
for name, c in log["checks"].items():
    print(f"{name}: rho = {c['rho']}, p = {c['p_value']}, n = {c['n']} -> {c['verdict']}")

### If the verdict is FAIL

The plan allows one escalation: a 20-epoch proxy. Uncomment and run the cell below. The finished medium runs are
reused from the log, so only the eight new proxy runs are trained, and the new rho is added to `checks` next to the
old one.

In [ ]:
# !python src/proxy_check.py --device 0 --proxy-epochs 20